# check_exact: verificación exacta de la geometría generada

Para cada escalón evalúa el espesor de PLA en el centro del escalón y lo compara con el diseño (espesor del escalón k = (N-k)·c75); para las láminas compara el espesor total.

**Requiere** `check_geom.py` en la carpeta del notebook (se importa igual que en el script original).

## 1. Imports

In [31]:
import sys, re, glob, os, importlib.util
import numpy as np

## 2. Parámetros
Carpetas de casos a verificar (en el script original eran los argumentos de la línea de comandos).

In [32]:
#args = sorted(glob.glob("casos/*"))
proyecto = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
args = sorted(glob.glob(os.path.join(proyecto, "casos", "*")))
here = os.getcwd()          # carpeta donde esta check_geom.py
args

['c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\LEEME.md',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N10',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N11',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12_res025um',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12_res050um',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12_res100um',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\ref_N9',
 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\slab_t00

## 3. Importar `check_geom.py`
`check_geom.py` procesa `sys.argv[1:]` al importarse, por eso se recorta `sys.argv` (importante en Jupyter, donde `sys.argv` trae los argumentos del kernel).

In [33]:
#ESTO ES SI LO CORREMOS COMO .PY
#sys.argv = sys.argv[:1]
#spec = importlib.util.spec_from_file_location("cg", os.path.join(here, "check_geom.ipynb"))
#cg = importlib.util.module_from_spec(spec); spec.loader.exec_module(cg)

In [34]:
#ESTO ES SI LO CORREMOS COMO .IPYNB

import nbformat
from types import SimpleNamespace

nb = nbformat.read(os.path.join(here, "check_geom.ipynb"), as_version=4)

cg = SimpleNamespace()

for cell in nb.cells:
    if cell.cell_type == "code":
        exec(cell.source, cg.__dict__)

## 4. Funciones y datos de referencia

In [35]:
ORIG = [0.04, 0.04, 0.05, 0.05, 0.05, 0.06, 0.07, 0.09, 0.18]
def thick(su, x):
    surf, sets, val = cg.load(su); t = open(su).read().replace('\r', '')
    c75 = val('c75'); c77 = val('c77')
    top = val(re.search(r'^1629\s+%pin_TR%\s+PZ\s+(.*?)\s*(\$|$)', t, flags=re.M).group(1))
    zz = np.linspace(c77 + 1e-9, top - 1e-9, 4000)
    return np.mean([cg.inside(sets, surf, x, z) for z in zz]) * (top - c77), c75, val('c71')

## 5. Verificación caso por caso

In [36]:
ok = True
for d in [x for x in args if os.path.isdir(x)]:
    name = os.path.basename(d.rstrip('/'))
    su = [s for s in glob.glob(d + '/dat/Su_ridge_filter_*.inp') if (not s.endswith('_9.inp')) or name == 'ref_N9'][0]
    desc = glob.glob(d + '/ridge_*.txt'); lam = glob.glob(d + '/lamina_*.txt')
    if lam:
        n = int(re.search(r'flat(\d+)', su).group(1)); exp = max(n, 1) * 0.02
        th1, _, c71 = thick(su, 0.3); th2, _, _ = thick(su, 0.01)
        good = abs(th1 - exp) < 1e-3 and abs(th2 - exp) < 1e-3; ok &= good
        print("%-14s lamina n=%-2d espesor %.3f cm (esperado %.3f)%s  %s" % (name, n, th1, exp, "  [material AIRE]" if n == 0 else "", "OK" if good else "ERROR"))
        continue
    if desc:
        txt = open(desc[0]).read().split('espesor 0):')[-1]
        w = [float(x) for x in re.findall(r'float64\(([0-9.]+)\)', txt)] or [float(x) for x in re.findall(r'[0-9]+\.[0-9]+|[0-9]+e-?[0-9]+', txt)]
    else:
        w = ORIG
    N = len(w); cum = 0.0; bad = []
    for k, wk in enumerate(w, 1):
        xc = cum + wk / 4            # centro del escalon k en x>0: tramo [cum, cum+wk/2]
        th, c75, c71 = thick(su, xc); cum += wk / 2
        if abs(th - (N - k) * c75) > 1e-3: bad.append((k, round(th, 3), round((N - k) * c75, 3)))
    ok &= not bad
    print("%-14s N=%-2d periodo=%.4f cm  %s %s" % (name, N, sum(w), "OK" if not bad else "ERROR", bad if bad else ""))

N10            N=10 periodo=0.6300 cm  OK 
N11            N=11 periodo=0.6300 cm  OK 
N12            N=12 periodo=0.6300 cm  OK 
N12_res025um   N=12 periodo=0.6300 cm  OK 
N12_res050um   N=12 periodo=0.6300 cm  OK 
N12_res100um   N=12 periodo=0.6300 cm  OK 
ref_N9         N=9  periodo=0.6300 cm  OK 
slab_t00       lamina n=0  espesor 0.020 cm (esperado 0.020)  [material AIRE]  OK
slab_t02       lamina n=1  espesor 0.020 cm (esperado 0.020)  OK
slab_t04       lamina n=2  espesor 0.040 cm (esperado 0.040)  OK
slab_t06       lamina n=3  espesor 0.060 cm (esperado 0.060)  OK
slab_t08       lamina n=4  espesor 0.080 cm (esperado 0.080)  OK
slab_t10       lamina n=5  espesor 0.100 cm (esperado 0.100)  OK
slab_t12       lamina n=6  espesor 0.120 cm (esperado 0.120)  OK
slab_t14       lamina n=7  espesor 0.140 cm (esperado 0.140)  OK
slab_t16       lamina n=8  espesor 0.160 cm (esperado 0.160)  OK
slab_t18       lamina n=9  espesor 0.180 cm (esperado 0.180)  OK
slab_t20       lamina n=10 espes

In [37]:
print("TODO OK" if ok else "HAY ERRORES")

TODO OK


In [38]:
print("Casos encontrados:", len(args))
print(args)

Casos encontrados: 20
['c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\LEEME.md', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N10', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N11', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12_res025um', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12_res050um', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\N12_res100um', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\casos\\ref_N9', 'c:\\Users\\Usuario\\Desktop\\Basili Rodriguez Labo 7\\Laboratory-VII\\phits_ridge_casos_v2\\c

In [39]:
caso = os.path.join(proyecto, "casos", "N12_res025um")

for raiz, carpetas, archivos in os.walk(caso):
    nivel = raiz.replace(caso, "").count(os.sep)
    if nivel <= 2:
        print(os.path.basename(raiz) or "N12_res025um")
        for archivo in archivos:
            print("    ", archivo)

N12_res025um
     80214.machines
     80214.mpd.hosts
     80214.slurm.err
     80214.slurm.out
     compartida.slurm
     phits.in
     principal.inp
     principal_rwt.inp
     ridge_N12r25.txt
analisis
     dosis.ipynb
     dvh_sin_atenuador.ipynb
     espectro.ipynb
     sobpp.ipynb
     tracks.ipynb
dat
     Ce_atenuador.inp
     Ce_dispersor1.inp
     Ce_dispersor2.inp
     Ce_fondo.inp
     Ce_fondo_mylar.inp
     Ce_hamster.inp
     Ce_iman.inp
     Ce_oclusor.inp
     Ce_posicionador.inp
     Ce_ridge_filter_9.inp
     Ce_ridge_filter_N12r25.inp
     Ce_viga.inp
     Su_atenuador.inp
     Su_dispersor1.inp
     Su_dispersor2.inp
     Su_fondo.inp
     Su_fondo_mylar.inp
     Su_hamster.inp
     Su_iman.inp
     Su_oclusor.inp
     Su_posicionador.inp
     Su_ridge_filter_9.inp
     Su_ridge_filter_N12r25.inp
     Su_viga.inp
     volumes.inp
out
     batch.out
     bragg_pouch_r.eps
     bragg_pouch_r.out
     bragg_pouch_z.eps
     bragg_pouch_z.out
     bragg_pouch_z_anillos

In [40]:
ridge_ref = os.path.join(caso, "ridge_N12r25.txt")

with open(ridge_ref, "r") as f:
    print(f.read())

N=12 escalones, espesor max=0.220 cm, periodo=0.6300 cm
anchos completos (cm), escalon 1 (grueso) -> N (espesor 0): [np.float64(0.025), np.float64(0.0275), np.float64(0.0275), np.float64(0.03), np.float64(0.0325), np.float64(0.035), np.float64(0.04), np.float64(0.045), np.float64(0.0525), np.float64(0.065), np.float64(0.0975), np.float64(0.1525)]



In [41]:
generado = os.path.join(here, "dat_NNLS12_res025um")

print(os.listdir(generado))

['Ce_ridge_filter_NNLS12_res025um.inp', 'Su_ridge_filter_NNLS12_res025um.inp']
